In [ ]:
import pandas as pd
from pathlib import Path

from helpers.met_data import (
    fetch_met_query,
    validate_urls_parallel,
    show_gallery,
)

In [ ]:
CACHE_CSV = Path("local/assets/sample_csv/met.csv")

CACHE_CSV.parent.mkdir(parents=True, exist_ok=True)

---
# Dataset: Pinturas del MET

Usamos la **API pública del MET** (Metropolitan Museum of Art, NYC) — sin API key, sin costo,
links estables.

El flujo:
```
MET API → IDs por tema → detalles de cada obra → filtrar con imagen pública
    → validar URLs en paralelo → dataset limpio de ~50 pinturas
```

In [ ]:
# Temas → cuántas obras de cada uno
TEMAS = {
    "impressionism": 10,
    "landscape nature": 10,
    "portrait": 10,
    "childhood children": 10,
    "dance music": 10,
    "armor knights": 10,
    "battle war": 10,
    "mythology": 10,
    "magic spells": 10,
    "cats dogs": 10,
    "jewelry fashion": 10
}

print("Consultando MET API por tema...")
rows = []
for tema, n in TEMAS.items():
    obras = fetch_met_query(tema, n_target=n)
    print(f"  {tema:25s} → {len(obras):2d} obras")
    rows.extend(obras)

In [ ]:
df_raw = pd.DataFrame(rows)
print(f"\nTotal antes de validar URLs: {len(df_raw)} obras")

In [ ]:
print(f"Validando {len(df_raw)} URLs en paralelo...")
validas = validate_urls_parallel(df_raw["image_url"].tolist())

df_raw["url_ok"] = validas
df = df_raw[df_raw["url_ok"]].drop(columns=["url_ok"]).reset_index(drop=True)

print(f"URLs válidas:  {sum(validas)}/{len(validas)}")
print(f"URLs con 404:  {len(validas) - sum(validas)}")
print(f"\nDataset final: {len(df)} obras")
print(df["style"].value_counts().to_string())

In [ ]:
# Guardar el CSV limpio para no repetir el fetch en clase
df.to_csv(CACHE_CSV, index=False)
print(f"✅ CSV guardado: {CACHE_CSV}  ({len(df)} obras)")
df.head()

In [ ]:
show_gallery(df, n=9)